# 📚 Literature Discovery Pipeline

Step-by-step notebook for discovering papers you may have missed.

**Pipeline:**
1. **Configure** — set your `.bib` file and options
2. **Explore** — resolve corpus, build co-citation graph, score candidates, fetch OpenAlex keywords
3. **Enrich** — fill missing abstracts (SS batch → OpenAlex → CrossRef)
4. **Search** — keyword/filter/semantic search over the full cache
5. **Visualize** — six interactive outputs (network, timeline, fields, score table, gap analysis, quadrant chart)
6. **Benchmark** *(optional)* — measure how well the tool rediscovers held-out papers

**Scoring overview (Step 2):**

| Signal | What it measures | Notes |
|--------|-----------------|-------|
| Co-citation | How often the candidate appears in corpus reference/citation lists | Percentile-normalized |
| Citations | Overall citation count | Percentile-normalized |
| Recency | How new the paper is relative to your corpus median year | Adaptive window |
| Field overlap | Fraction of the paper's fields matching your corpus | Proportional bonus |
| Semantic similarity | Cosine similarity to corpus abstracts | sentence-transformers or TF-IDF fallback |
| Keyword text match | Keyword hit in title/abstract | Only applied when `KEYWORDS` is set |
| OpenAlex keyword/topic | Overlap with OA-supplied keywords | Only applied when `KEYWORDS` is set |

**Directional co-citation:** the tool distinguishes *foundational* papers (your corpus cites them) from *follow-on* papers (they cite your corpus). Both count toward scoring; the breakdown is shown in the score table and network layout.

> Run cells top-to-bottom. Each step is independent after Step 2.

---
## ⚙️ Step 0 — Configuration
Edit the variables below, then run this cell. Everything else uses these.

In [ ]:
from pathlib import Path
from types import SimpleNamespace

# ── Change these ──────────────────────────────────────────────────────────────
BIB_FILE     = "references.bib"   # path to your .bib file
PDF_DIR      = None               # set to a folder path to use PDFs instead of bib

# Scoring options
KEYWORDS     = []                 # focus keywords — influence scoring in three ways:
                                  #   1. title/abstract text match  (kw_score, weight 0.15)
                                  #   2. OpenAlex keyword/topic overlap (oa_kw_score, weight 0.15)
                                  #   3. proportional keyword bonus applied in base score
                                  # e.g. ["replicability", "open data", "preregistration"]

PRESET       = "balanced"         # weight preset for base score:
                                  #   "balanced"          co_cite 35% · citations 30% · recency 20% · field 15%
                                  #   "highly-cited"      co_cite 20% · citations 60% · recency 10% · field 10%
                                  #   "recent"            co_cite 25% · citations 15% · recency 50% · field 10%
                                  #   "interdisciplinary" co_cite 35% · citations 25% · recency 15% · field 25%

CLUSTER      = None               # topic clustering: None = skip, 0 = auto-detect k, N = fixed k

# Output / infrastructure
OUT_DIR      = "outputs"          # root output folder
TOP_N        = 50                 # top N candidates shown in network visualization
ENRICH_LIMIT = 0                  # max individual OA/CR abstract API calls (0 = unlimited)
# ─────────────────────────────────────────────────────────────────────────────

# Automatically derived — do not edit
if BIB_FILE:
    PROJ_NAME  = Path(BIB_FILE).stem
    CACHE_PATH = Path(BIB_FILE).parent / f"{PROJ_NAME}_cache.json"
elif PDF_DIR:
    PROJ_NAME  = Path(PDF_DIR.rstrip("/\\")).stem
    CACHE_PATH = Path(PDF_DIR).parent / f"{PROJ_NAME}_cache.json"
else:
    raise ValueError("Set BIB_FILE or PDF_DIR above")

print(f"Project   : {PROJ_NAME}")
print(f"Cache     : {CACHE_PATH}")
print(f"Outputs   : {OUT_DIR}/{PROJ_NAME}/")
print(f"Preset    : {PRESET}")
if KEYWORDS:
    print(f"Keywords  : {KEYWORDS}")
if CLUSTER is not None:
    print(f"Cluster   : {'auto-k' if CLUSTER == 0 else f'k={CLUSTER}'}")

---
## 🔍 Step 1 — Explore

Resolves each paper via the **Semantic Scholar batch API** (corpus papers with DOIs are batch-fetched in groups of 100; title-only papers fall back to individual searches). Then builds the co-citation graph and scores every candidate.

**Scoring pipeline:**

1. **Co-citation graph** — counts how often each outside paper appears in your corpus's reference and citation lists, broken down into:
   - *Foundational* (`ref_count`): corpus papers that cite the candidate
   - *Follow-on* (`cite_count`): corpus papers the candidate cites

2. **Base score** — percentile-normalized blend of co-citation, citation count, recency, and field overlap (weights set by `PRESET`). Using percentiles means one highly-cited outlier won't drown out everything else.
   - **Field bonus** — proportional to how many of the candidate's fields match your corpus: `1.0 + 0.5 × (overlap / total_fields)`
   - **Recency** — decay window derived from your corpus's *median* publication year, so newer fields aren't unfairly penalized
   - **Keyword bonus** — proportional overlap with user keywords, applied when `KEYWORDS` is set

3. **Near-duplicate filter** — groups candidates by first author + title word overlap (≥ 85% Jaccard). Only the higher-cited version of near-identical papers (e.g. preprint + journal) is kept.

4. **Final blend** (top 200 candidates re-ranked):

| When | Formula |
|------|---------|
| Keywords provided | `0.55 × base + 0.15 × sim + 0.15 × kw_text + 0.15 × oa_kw` |
| No keywords | `0.80 × base + 0.20 × sim` |

   - `sim` — semantic similarity to corpus abstracts (sentence-transformers `all-MiniLM-L6-v2` if installed, TF-IDF otherwise). Embeddings are cached so subsequent runs skip the model.
   - `kw_text` — keyword match score against candidate title/abstract
   - `oa_kw` — OpenAlex keyword/topic overlap (batch-fetched from OA after initial scoring)

> **API calls**: corpus resolved once then cached. Concurrent resolution (3 threads) with a shared SS rate-limiter keeps calls under the API limit. Re-runs with a warm cache make zero API calls.

In [ ]:
from core.explorer import run as run_explorer

explorer_result = run_explorer(SimpleNamespace(
    bib      = BIB_FILE,
    pdf_dir  = PDF_DIR,
    out      = OUT_DIR,
    cache    = str(CACHE_PATH),
    keywords = KEYWORDS or None,
    preset   = PRESET,
    cluster  = CLUSTER,
))

print(f"\n✅ Core papers      : {explorer_result['core']}")
print(f"   Periphery papers : {explorer_result['periphery']}")
print(f"   Report saved to  : {explorer_result['out_dir']}/")

---
## 🌐 Step 2 — Enrich abstracts
Fills in `null` abstracts for every paper in the cache (corpus + embedded refs/citations).
Uses Semantic Scholar batch API first (fastest), then OpenAlex, then CrossRef.

> **Safe to skip** if abstracts are already populated. Idempotent — re-running is harmless.
> The original cache is backed up as `<cache>.bak` before any writes.

In [ ]:
from core.enrich import run as run_enrich

# Dry run first — shows how many abstracts are missing, no API calls
run_enrich(SimpleNamespace(
    cache   = str(CACHE_PATH),
    limit   = 0,
    dry_run = True,
    sources = ["ss", "cr"],    # OA excluded by default (low hit-rate); add "oa" to enable
))

In [ ]:
# Actually fetch — comment out if you want to skip enrichment
enrich_result = run_enrich(SimpleNamespace(
    cache   = str(CACHE_PATH),
    limit   = ENRICH_LIMIT,
    dry_run = False,
    sources = ["ss", "cr"],    # OA excluded by default (low hit-rate); add "oa" to enable
))

print(f"\n✅ Nodes filled   : {enrich_result['nodes_filled']}")
print(f"   Still missing  : {enrich_result['still_missing']}")

---
## 🔎 Step 3 — Search
Keyword, author, and filter-based search over the full cached paper pool.
No API calls — works entirely from the local cache.

In [ ]:
from core.search import run as run_search

# ── Edit search parameters here ───────────────────────────────────────────────
SEARCH_KEYWORDS  = ["open science"]   # plain words or "quoted phrases"
SEARCH_AUTHOR    = None               # e.g. "Nosek" — substring, case-insensitive
SEARCH_FIELD     = None               # e.g. "psychology"
SEARCH_YEAR_MIN  = None               # e.g. 2015
SEARCH_YEAR_MAX  = None               # e.g. 2024
SEARCH_MIN_CITES = None               # e.g. 50
SEARCH_MAX_CITES = None
SEARCH_IN        = "all"              # "all", "title", or "abstract"
SEARCH_REGEX     = False              # set True to use regex patterns as keywords
SEARCH_TOP       = 20
SEARCH_SORT      = "relevance"        # "relevance", "citations", or "year"
SEARCH_VERBOSE   = True               # show abstract snippets
# ─────────────────────────────────────────────────────────────────────────────

results = run_search(SimpleNamespace(
    cache         = str(CACHE_PATH),
    keywords      = SEARCH_KEYWORDS,
    author        = SEARCH_AUTHOR,
    field         = SEARCH_FIELD,
    year_min      = SEARCH_YEAR_MIN,
    year_max      = SEARCH_YEAR_MAX,
    min_citations = SEARCH_MIN_CITES,
    max_citations = SEARCH_MAX_CITES,
    search_in     = SEARCH_IN,
    regex         = SEARCH_REGEX,
    top           = SEARCH_TOP,
    sort          = SEARCH_SORT,
    verbose       = SEARCH_VERBOSE,
    mode          = "any",
    export        = None,
    stats         = False,
))

---
### 🔬 Step 3b — Advanced search options

New filters: `--type`, `--venue`, `--new-only`, `--exclude`, `--sort trending`, `--semantic`

In [ ]:
from core.search import run as run_search

# ── Advanced search parameters ────────────────────────────────────────────────
# Filter by detected paper type (heuristic on abstract)
SEARCH_TYPE    = None      # None | "meta-analysis" | "review" | "empirical" |
                           #        "methods" | "theoretical" | "replication"

# Filter by venue/journal substring
SEARCH_VENUE   = None      # e.g. "Nature", "PNAS", "Psychological Science"

# Only show newly discovered papers (not in your original bib)
SEARCH_NEW_ONLY = False

# Exclude papers containing any of these terms
SEARCH_EXCLUDE  = None     # e.g. ["preprint", "blog", "commentary"]

# Semantic (TF-IDF) search — finds conceptually similar papers even without exact keyword match
SEARCH_SEMANTIC = None     # e.g. "open science reform practices"

# Find papers similar to a given title (uses that paper's abstract as the semantic query)
SEARCH_SEMANTIC_LIKE = None  # e.g. "The Role of Preregistration in Psychology"

# Sort by citation velocity (citations / years since publication)
SEARCH_SORT_TRENDING = False
# ─────────────────────────────────────────────────────────────────────────────

results_adv = run_search(SimpleNamespace(
    cache         = str(CACHE_PATH),
    keywords      = ["open science"],   # change or set to []
    author        = None,
    field         = None,
    year_min      = None,
    year_max      = None,
    min_citations = None,
    max_citations = None,
    search_in     = "all",
    regex         = False,
    top           = 20,
    sort          = "trending" if SEARCH_SORT_TRENDING else "relevance",
    verbose       = True,
    mode          = "any",
    export        = None,
    export_format = "bib",
    stats         = False,
    new_only      = SEARCH_NEW_ONLY,
    exclude       = SEARCH_EXCLUDE,
    type          = SEARCH_TYPE,
    venue         = SEARCH_VENUE,
    semantic      = SEARCH_SEMANTIC,
    semantic_like = SEARCH_SEMANTIC_LIKE,
))

In [ ]:
# Export search results to BibTeX (optional)
# Uncomment and set a filename to export

# from core.search import run as run_search
# run_search(SimpleNamespace(
#     cache=str(CACHE_PATH), keywords=SEARCH_KEYWORDS,
#     author=SEARCH_AUTHOR, field=SEARCH_FIELD,
#     year_min=SEARCH_YEAR_MIN, year_max=SEARCH_YEAR_MAX,
#     min_citations=SEARCH_MIN_CITES, max_citations=SEARCH_MAX_CITES,
#     search_in=SEARCH_IN, regex=SEARCH_REGEX,
#     top=SEARCH_TOP, sort=SEARCH_SORT, verbose=False,
#     mode="any", stats=False,
#     export="search_results.bib",   # ← filename here
# ))

---
## 📊 Step 4 — Visualize

Generates four interactive outputs from the cache:

| Output | Description |
|--------|-------------|
| **Network** (HTML) | GPU-accelerated co-citation graph via **cosmos.gl** (WebGL). Corpus = blue circles, candidates = coloured circles sized by citation count. Physics runs live on the GPU — hover a node to highlight its neighbours, click to open its DOI. Requires an internet connection to load the cosmos.gl library. |
| **Timeline** (HTML) | Year × citation scatter. Hover for abstract snippet; click to open DOI. |
| **Fields** (PNG) | Field distribution bar chart. Green = overlap with your corpus. |
| **Score table** (HTML) | Filterable table with per-column range sliders, percentile ranks, abstract previews, and CSV export. |

In [ ]:
from types import SimpleNamespace
from core.viz import run as run_viz
import webbrowser

viz_paths = run_viz(SimpleNamespace(
    cache        = str(CACHE_PATH),
    top          = TOP_N,
    out          = OUT_DIR,
    no_network   = False,
    no_timeline  = False,
    no_fields    = False,
    no_table     = False,
))

print("\nOutput files:")
for name, path in viz_paths.items():
    print(f"  {name:<10}: {path}")

In [ ]:
# Open visualizations in your browser
if "network" in viz_paths:
    webbrowser.open(viz_paths["network"].resolve().as_uri())
if "timeline" in viz_paths:
    webbrowser.open(viz_paths["timeline"].resolve().as_uri())

---
## 🧪 Step 5 — Benchmark *(optional)*

Measures how well the tool rediscovers papers you already know about. Randomly removes a fraction of your `.bib` entries, runs the explorer on the remaining papers, then checks if the removed papers appear in the top-K recommendations.

Run this to:
- Compare presets (`balanced` vs `recent` vs `highly-cited`) on your specific corpus
- Check whether adding `KEYWORDS` improves recall
- Verify that code changes haven't degraded quality

Reports **recall@10**, **recall@25**, **recall@50** averaged over multiple trials.

> Uses live API calls by default. Set `BENCHMARK_DRY_RUN = True` to reuse cached data (fast, but only meaningful if you already have a warm cache).

In [ ]:
# ── Benchmark configuration ───────────────────────────────────────────────────
BENCHMARK_TRIALS   = 10     # number of random hold-out trials
BENCHMARK_HOLDOUT  = 0.1    # fraction of papers to remove per trial (0.1 = 10%)
BENCHMARK_TOP      = 50     # largest K to report recall@K for
BENCHMARK_DRY_RUN  = False  # True = skip API calls (warm cache only)
BENCHMARK_SEED     = 42     # set to None for non-reproducible random trials
# ─────────────────────────────────────────────────────────────────────────────

import sys, random
sys.path.insert(0, ".")
from benchmark.held_out import run_trial, _parse_bib_entries

bib_text    = (BIB_FILE and open(BIB_FILE).read()) or ""
all_entries = _parse_bib_entries(bib_text)
print(f"Loaded {len(all_entries)} bib entries")

if BENCHMARK_SEED is not None:
    random.seed(BENCHMARK_SEED)

from pathlib import Path
import time

ks           = sorted({10, 25, 50, BENCHMARK_TOP})
recall_sums  = {k: 0.0 for k in ks}
n_completed  = 0
out_root     = Path("benchmark_tmp")

for trial in range(1, BENCHMARK_TRIALS + 1):
    print(f"\n── Trial {trial}/{BENCHMARK_TRIALS} ──────────────────────────────")
    t0     = time.time()
    result = run_trial(all_entries, BENCHMARK_HOLDOUT, BENCHMARK_TOP,
                       out_root, BENCHMARK_DRY_RUN, PRESET, KEYWORDS or None)
    elapsed = time.time() - t0

    if result is None:
        print("  Trial failed — skipped")
        continue

    n_completed += 1
    held_out = result["held_out"]
    print(f"  Held-out: {held_out} | elapsed: {elapsed:.1f}s")
    for k in ks:
        found  = result["found"][k]
        recall = found / held_out if held_out else 0.0
        recall_sums[k] += recall
        print(f"  recall@{k:<4}: {found}/{held_out}  ({recall:.1%})")

print(f"\n{'─'*50}")
print(f"Results ({n_completed}/{BENCHMARK_TRIALS} trials)")
for k in ks:
    mean_recall = recall_sums[k] / n_completed if n_completed else 0.0
    print(f"  Mean recall@{k:<4}: {mean_recall:.1%}")

---
## 🚀 Run everything at once
Equivalent to `python run.py --bib your_file.bib`

In [ ]:
# Uncomment to run the full pipeline in one shot (equivalent to run.py)
# import subprocess, sys
# cmd = [
#     sys.executable, "run.py",
#     "--bib",    BIB_FILE,
#     "--preset", PRESET,
#     "--top",    str(TOP_N),
#     "--out",    OUT_DIR,
# ]
# if KEYWORDS:
#     cmd += ["--keywords"] + KEYWORDS
# if CLUSTER is not None:
#     cmd += ["--cluster", str(CLUSTER)]
# if ENRICH_LIMIT:
#     cmd += ["--enrich-limit", str(ENRICH_LIMIT)]
# subprocess.run(cmd)